# 01 — Data Collection & Feature Matrix Construction

This notebook loads raw data from the client Excel and O*NET database,
then builds the feature matrix used for clustering and network analysis.

**Owner**: Person A

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
from src.onet_data import get_valid_codes, build_feature_matrix, load_related_occupations, load_job_zones
from src.data_cleaning import (
    load_agc_members, load_dot_prequal, load_apprenticeships,
    load_community_college, load_umaine, load_onet_codes
)

EXCEL = '../data/raw/Career_Drive_Project_Data_Sources.xlsx'
ONET_DB = '../data/raw/onet_db'

## 1. Extract valid O*NET codes from Excel

The Excel sheet has 28 rows but only 21 unique valid codes after filtering out
section headers, blanks, and duplicates. One code (13-1082.00) has no data in
the O*NET database, leaving us with 20 usable occupations.

In [ ]:
codes = get_valid_codes(EXCEL)
print(f'Valid unique codes: {len(codes)}')
for c in codes:
    print(f'  {c}')

## 2. Build feature matrix from O*NET database

In [ ]:
features = build_feature_matrix(ONET_DB, codes)
print(f'Feature matrix: {features.shape}')
print(f'Missing values: {features.drop(columns="Title").isna().sum().sum()}')
features[['Title']].head(10)

In [ ]:
features.to_csv('../data/processed/occupation_features.csv')
print('Saved occupation_features.csv')

## 3. Load Related Occupations and Job Zones

In [ ]:
# Related occupations (edges for network graph)
edges = load_related_occupations(ONET_DB, list(features.index))
print(f'Related occupation edges: {len(edges)}')
edges.to_csv('../data/processed/related_occupations.csv', index=False)

# Job zones (complexity labels for validation)
jz = load_job_zones(ONET_DB, list(features.index))
print(f'Job zones: {len(jz)}')
jz.to_csv('../data/processed/job_zones.csv', index=False)
print(jz.to_string(index=False))

## 4. Load client Excel data

In [ ]:
agc = load_agc_members(EXCEL)
print(f'AGC Members: {len(agc)}')
print(agc['type'].value_counts())

In [ ]:
dot = load_dot_prequal(EXCEL)
print(f'DOT Prequal: {len(dot)}')

In [ ]:
apprentice = load_apprenticeships(EXCEL)
print(f'Apprenticeships: {len(apprentice)}')
apprentice.head()

In [ ]:
cc = load_community_college(EXCEL)
umaine = load_umaine(EXCEL)
print(f'Community College programs: {len(cc)}')
print(f'UMaine programs: {len(umaine)}')

## 5. Save all cleaned datasets

In [ ]:
agc.to_csv('../data/processed/agc_members.csv', index=False)
dot.to_csv('../data/processed/dot_prequal.csv', index=False)
apprentice.to_csv('../data/processed/apprenticeships.csv', index=False)
cc.to_csv('../data/processed/community_college.csv', index=False)
umaine.to_csv('../data/processed/umaine_programs.csv', index=False)
print('All datasets saved to data/processed/')